# CP6 - IA: Treinando uma LLM Local com GPU (Fine-tuning LoRA)

**Problema:** Assistente de perguntas e respostas (Q&A) em português sobre a FIAP e conceitos de Inteligência Artificial.

**Modelo base:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

**Técnica:** Fine-tuning com QLoRA (quantização 4-bits + LoRA)

## 1. Informações da GPU

In [1]:
!nvidia-smi

Wed Oct 14 15:51:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print('CUDA disponivel:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Versao do CUDA (PyTorch):', torch.version.cuda)
    print('Memoria total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('ATENCAO: GPU nao detectada.')

CUDA disponivel: True
GPU: Tesla T4
Versao do CUDA (PyTorch): 13.0
Memoria total (GB): 15.64


## 2. Dependências

In [3]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 14.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 3. Dataset

In [4]:
from google.colab import files
import os
if not os.path.exists('qa_fiap.jsonl'):
    uploaded = files.upload()

In [5]:
from datasets import load_dataset

dataset = load_dataset('json', data_files='qa_fiap.jsonl', split='train')
print('Numero de exemplos:', len(dataset))
print('Exemplo:', dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Numero de exemplos: 30
Exemplo: {'instruction': 'O que é a FIAP?', 'input': '', 'output': 'A FIAP (Faculdade de Informática e Administração Paulista) é uma instituição de ensino superior brasileira focada em tecnologia, inovação e negócios. Oferece cursos de graduação, pós-graduação e MBA voltados principalmente para a área de tecnologia da informação.'}


## 4. Formatação do dataset no template de chat

In [6]:
from transformers import AutoTokenizer

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = 'Voce e um assistente educacional da FIAP. Responda em portugues de forma clara e objetiva.'

def format_example(ex):
    pergunta = ex['instruction']
    if ex.get('input'):
        pergunta += '\n' + ex['input']
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': pergunta},
        {'role': 'assistant', 'content': ex['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {'text': text}

dataset = dataset.map(format_example)
print(dataset[0]['text'])

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

<|system|>
Voce e um assistente educacional da FIAP. Responda em portugues de forma clara e objetiva.</s>
<|user|>
O que é a FIAP?</s>
<|assistant|>
A FIAP (Faculdade de Informática e Administração Paulista) é uma instituição de ensino superior brasileira focada em tecnologia, inovação e negócios. Oferece cursos de graduação, pós-graduação e MBA voltados principalmente para a área de tecnologia da informação.</s>



## 5. Modelo com quantização 4-bits (QLoRA)

In [19]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 6. Treinamento

In [20]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./tinyllama-fiap',
    num_train_epochs=30,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    fp16=False,
    bf16=False,
    logging_steps=5,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    save_strategy='epoch',
    report_to='none',
    dataset_text_field='text',
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

Step,Training Loss
5,1.525469
10,0.683187
15,0.253516
20,0.108056
25,0.058238
30,0.042547
35,0.039199
40,0.037473
45,0.033140
50,0.033543


TrainOutput(global_step=120, training_loss=0.1338565188149611, metrics={'train_runtime': 199.1546, 'train_samples_per_second': 4.519, 'train_steps_per_second': 0.603, 'total_flos': 804433098350592.0, 'train_loss': 0.1338565188149611, 'epoch': 30.0})

In [21]:
model.save_pretrained('tinyllama-fiap-lora')
tokenizer.save_pretrained('tinyllama-fiap-lora')
print('Modelo salvo em tinyllama-fiap-lora/')

# Prepara o modelo para INFERENCIA: desliga o gradient checkpointing (usado so no treino)
# e religa o cache. Sem isso a geracao sai como texto incoerente.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()
print('Modelo pronto para gerar respostas.')

Modelo salvo em tinyllama-fiap-lora/
Modelo pronto para gerar respostas.


## 7. Avaliação das respostas

In [22]:
from transformers import pipeline

def responder(pergunta, max_new_tokens=160, temperature=0.3):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': pergunta},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    resposta = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return resposta.strip()

perguntas_teste = [
    'O que e a FIAP?',
    'O que e LoRA no treinamento de LLMs?',
    'Para que serve a GPU no treinamento de IA?',
    'O que e um Challenge na FIAP?',
    'Como organizar meu tempo de estudo?',
]

for p in perguntas_teste:
    print('='*70)
    print('PERGUNTA:', p)
    print('RESPOSTA:', responder(p))
    print()

[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PERGUNTA: O que e a FIAP?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPOSTA: A FIAP (Faculdade de Informática e Administração Paulista) é uma instituição de ensino superior brasileira focada em tecnologia, inovação e negócios. Oferece cursos de graduação, pós-graduação e MBA voltados principalmente para a área de tecnologia da informação.

PERGUNTA: O que e LoRA no treinamento de LLMs?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPOSTA: LoRA (Low-Rank Adaptation) é uma técnica de fine-tuning eficiente que congela os pesos originais do modelo e treina apenas pequenas matrizes adicionais. Isso reduz drasticamente a quantidade de memória e processo necessários, permitindo treinar modelos grandes em GPUs modestas.

PERGUNTA: Para que serve a GPU no treinamento de IA?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPOSTA: A GPU possui milhares de núcleos capazes de realizar cálculos em paralelo, o que acelera muito as operações matriciais usadas no treinamento de redes neurais. Uma tarefa que levaria horas na CPU pode levar minutos na GPU.

PERGUNTA: O que e um Challenge na FIAP?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPOSTA: Um Challenge é uma orientação prática e integradora durante o semestre. Eles valem nota e servem para acompanhar a evolução do aluno ao longo do curso, podendo ser individuais ou em grupo, teóricos ou práticos.

PERGUNTA: Como organizar meu tempo de estudo?
RESPOSTA: Para organizar seu tempo de estudo, revise o plano semestraal (semester plan) ou a biblioteca de arquivos do curso. Eles valem nota e servem para planejar bem antes de começar o trabalho e para seguir uma orientação durante o treinamento.



## 8. Comparação de temperatura

In [23]:
pergunta = 'O que e fine-tuning de um modelo de linguagem?'
for t in [0.2, 0.7, 1.0]:
    print('='*70)
    print(f'Temperatura = {t}')
    print(responder(pergunta, temperature=t))
    print()

[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temperatura = 0.2


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuning é o processo de pegar um modelo de linguagem já pré-treinado e continuar o treinamento com um conjunto de dados específico, para que ele se especialize em um domínio ou tarefa. Isso é mais barato e rápido do que treinar um modelo do zero.

Temperatura = 0.7


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuning é o processo de pegar um modelo de linguagem já pré-treinado e continuar o treinamento com um conjunto de dados específico, para que ele se especialize em um domínio ou tarefa. Isso é mais barato e rápido do que treinar um modelo do zero.

Temperatura = 1.0
Fine-tuning é o processo de pegar um modelo de linguagem já pré-treinado e continuamente treinar ou enxergar ele com novas instruções ou dados, para que ele se especialize em unidades, tarefas ou domínio específico. Isso é mais barato e rápido do que treinar um modelo do zero.



## 9. Exportar o adaptador LoRA

In [24]:
!zip -r tinyllama-fiap-lora.zip tinyllama-fiap-lora
from google.colab import files
files.download('tinyllama-fiap-lora.zip')

  adding: tinyllama-fiap-lora/ (stored 0%)
  adding: tinyllama-fiap-lora/tokenizer_config.json (deflated 46%)
  adding: tinyllama-fiap-lora/adapter_model.safetensors (deflated 22%)
  adding: tinyllama-fiap-lora/tokenizer.json (deflated 85%)
  adding: tinyllama-fiap-lora/README.md (deflated 65%)
  adding: tinyllama-fiap-lora/chat_template.jinja (deflated 60%)
  adding: tinyllama-fiap-lora/adapter_config.json (deflated 61%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>